In [ ]:
import os
from langchain.chat_models import init_chat_model
from typing import Literal
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        default="北京",
        description="城市"
    )


model = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_API_BASE_URL"),
)

unit: Literal["celsius", "fahrenheit"] = Field(
    default="celsius",
    description="气温单位"
)
include_forecast: bool = Field(
    default=False,
    description="是否包含未来五日天气预报"
)
print(WeatherInput())

In [ ]:
from langchain.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool


@tool("get_weather_and_forecast", parse_docstring=True)
def get_weather(city: str = "北京", if_forecast: bool = False):
    """
    查询当日天气，可以包含明日天气预报
    Args:
    city: 城市名称
    if_forecast: 是否包含明日天气预报
"""
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n明天要下雨"
    return res


print(convert_to_openai_tool(get_weather))
model_with_tools = model.bind_tools([get_weather])
messages = [HumanMessage("今天杭州天气如何？明天呢？")]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls
# 将工具调用的结果添加到消息列表中
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        # 返回值tool_msg类型是ToolMessage
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)
final_response = model_with_tools.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()


In [ ]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:


    """
    获取当日天气
    Args:
    city: 城市名称
    """
    return f'{city}当天晴朗'
model_with_tools = model.bind_tools([get_weather], tool_choice="none") #不让模型调用工具 可选参数none auto和required
# model_with_tools = model.bind_tools([get_weather1, get_weather2],tool_choice="get_weather2")
messages = [
    HumanMessage("今天北京天气如何？别瞎编")
]
response = model_with_tools.invoke(messages)
response.pretty_print()